# Neural Ninjas — Face Rating Regression (Google Colab Notebook)

**Team:** Neural Ninjas (Batch A3) | **Course:** ML Lab (AIP301)  
**Approach:** Classical Machine Learning (No CNNs, No Transfer Learning)

---
### Instructions:
1. Put the `neural_ninja` folder with `Images/` and `labels.txt` in your Google Drive (e.g., in `/content/drive/MyDrive/neural_ninja`).
2. Execute the cells below sequentially.

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Set your repository folder path
PROJECT_DIR = '/content/drive/MyDrive/neural_ninja'
os.chdir(PROJECT_DIR)
print('Active Working Directory:', os.getcwd())
print('Directory Contents:', os.listdir('.'))

### Step 2: Install Dependencies (OpenCV < 5 for Haar Cascades)

In [ ]:
!pip install -q numpy pandas scipy scikit-learn scikit-image 'opencv-python-headless>=4.8,<5' matplotlib joblib openpyxl tqdm

### Step 3: Verify Data Availability

In [ ]:
import os

images_path = os.path.join(PROJECT_DIR, 'Images')
labels_path = os.path.join(PROJECT_DIR, 'labels.txt')

valid_images = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
with open(labels_path, 'r') as f:
    labels_count = sum(1 for line in f if len(line.strip().split()) >= 2)

print(f'Total Image Files Found: {len(valid_images)}')
print(f'Total Label Entries:     {labels_count}')
assert len(valid_images) > 0, 'Error: No images found. Check your Drive directory.'
assert labels_count > 0, 'Error: labels.txt is empty.'

### Step 4: Run Preprocessing (Stages 1–4)
Runs image audit, dHash duplicate detection, group disjoint splits, and target definitions.

In [ ]:
!python preprocessing.py --data_dir Images --labels labels.txt --out out --jobs 2 --batch_size 16

### Step 5: Run Full Modelling & Evaluation Pipeline (Stages 5–10)
Uses `--resume` to seamlessly pick up from the saved preprocessing checkpoint.

In [ ]:
!python run_pipeline.py --data_dir Images --labels labels.txt --out out --jobs 2 --batch_size 16 --resume

### Step 6: Review Key Metrics

In [ ]:
import pandas as pd
from IPython.display import display

print('=== CROSS-VALIDATION SUMMARY (Task 8) ===')
cv_summary = pd.read_csv('out/task8_cv_summary.csv', index_col=0)
display(cv_summary[['MAE_mean', 'RMSE_mean', 'RMSE_std', 'Pearson_mean', 'n_features', 'fit_s_mean']].round(3))

print('\n=== FINAL TEST EVALUATION (Single Held-Out Test Set) ===')
test_results = pd.read_csv('out/test_results.csv')
display(test_results.round(4))

### Step 7: Visualize Generated Diagnostic Figures

In [ ]:
import os
from IPython.display import Image, display

figures = [
    'fig01_target_distribution.png',
    'fig05_leakage_check.png',
    'fig_pca_vs_kmeans.png',
    'fig08_metric_comparison.png',
    'fig_scoreboard_and_cost.png',
    'fig_learning_curves.png',
    'fig_performance_vs_dimension.png',
    'fig_feature_selection.png',
    'fig_confusion_matrices.png',
    'fig_residuals.png',
    'fig_error_gallery.png',
    'fig_group_and_range_errors.png'
]

for fig_name in figures:
    fig_path = os.path.join('out', fig_name)
    if os.path.exists(fig_path):
        print(f'\n--- Displaying: {fig_name} ---')
        display(Image(filename=fig_path, width=800))
    else:
        print(f'Figure not found: {fig_name}')

### Step 8: Test Inference with `predict.py`

In [ ]:
import glob, random

sample_files = glob.glob('Images/*.jpg')
if sample_files:
    sample_img = random.choice(sample_files)
    print(f'Testing inference on: {sample_img}')
    !python predict.py {sample_img} --bundle out/bundle.joblib
else:
    print('No image available to test.')